In [1]:
import os
import sys
import gc
import ctypes

os.environ["CUDA_VISIBLE_DEVICES"] = "1,5,6,7"
# =========================
# 0. 先读取命令行参数
# 用法:
# CUDA_VISIBLE_DEVICES=0 python train_dim.py 2
# CUDA_VISIBLE_DEVICES=1 python train_dim.py 3
# =========================
dim = 3  # notebook里默认先手动指定
print(f"Running latent dim = {dim}")

# =========================
# 1. 预加载 CUDA 11 库
# 注意：这一步必须在 import tensorflow 之前
# =========================
lib_path = "/home/yinghuazhang/miniconda3/envs/daes/lib"
libs = [
    "libcudart.so.11.0",
    "libcublas.so.11",
    "libcublasLt.so.11",
    "libcufft.so.10",
    "libcurand.so.10",
    "libcusolver.so.11",
    "libcusparse.so.11",
    "libcudnn.so.8",
]

print("--- 开始预加载 CUDA 11 库 ---")
for lib in libs:
    full_path = os.path.join(lib_path, lib)
    try:
        ctypes.CDLL(full_path)
        print(f"✅ {lib} 加载成功")
    except Exception as e:
        print(f"❌ {lib} 加载失败: {e}")

# =========================
# 2. 设置 GPU 可见性
# 最好不要在这里写死 1,5,6,7
# 而是由外部命令控制：
# CUDA_VISIBLE_DEVICES=0 python train_dim.py 2
# =========================
# 不在代码里强制写死单卡
# 由外部启动命令决定每个进程使用哪张GPU
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "Not set"))

# =========================
# 3. 再导入 TensorFlow
# =========================
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ 已成功为 {len(gpus)} 张显卡开启显存增长模式")
    except RuntimeError as e:
        print(f"⚠️ 设置显存增长时出错: {e}")

# =========================
# 4. 其余常规模块导入
# =========================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # 脚本模式下保存图片，不依赖显示器
import matplotlib.pyplot as plt

from joblib import load, dump
from tensorflow.keras import Model
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, Conv2DTranspose, MaxPooling2D,
    Flatten, UpSampling2D, Reshape
)
from tensorflow.keras.models import Sequential

from molmap.model import RegressionEstimator, MultiClassEstimator, MultiLabelEstimator
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.utils import shuffle
from molmap import dataset, MolMap, feature

Running latent dim = 3
--- 开始预加载 CUDA 11 库 ---
✅ libcudart.so.11.0 加载成功
✅ libcublas.so.11 加载成功
✅ libcublasLt.so.11 加载成功
✅ libcufft.so.10 加载成功
✅ libcurand.so.10 加载成功
✅ libcusolver.so.11 加载成功


✅ libcusparse.so.11 加载成功
✅ libcudnn.so.8 加载成功
CUDA_VISIBLE_DEVICES = 1,5,6,7


2026-03-06 17:54:08.040067: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]
✅ 已成功为 4 张显卡开启显存增长模式


/home/yinghuazhang/miniconda3/envs/daes/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yinghuazhang/miniconda3/envs/daes/lib/python3.9/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from keras.backend import set_session
from keras.backend import clear_session
from keras.backend import get_session
import tensorflow as tf
import gc
 
# Reset Keras Session
def reset_keras():
    sess = get_session()
    clear_session()
    sess.close()
    sess = get_session()
 
    try:
        del classifier # this is from global space - change this as you need
    except:
        pass
 
    print(gc.collect()) # if it does something you should see a number as output
 
    # use the same config as you used to create the session
    config = tf.compat.v1.ConfigProto()
    config.gpu_options.per_process_gpu_memory_fraction = 1
    config.gpu_options.visible_device_list = "0"
    set_session(tf.compat.v1.Session(config=config))



def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_history_plot(history_dict, save_path, title):
    df = pd.DataFrame(history_dict)
    ax = df.plot(figsize=(6, 4))
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

def save_reconstruction_plot(x_true, x_pred, save_path, n_show=10):
    fig = plt.figure(figsize=(20, 8))
    for i in range(n_show):
        ax = plt.subplot(4, 5, i + 1)
        ax.imshow(x_true[i], cmap="gray")
        ax.set_title(f"True {i}")
        ax.axis("off")

        ax = plt.subplot(4, 5, i + 1 + n_show)
        ax.imshow(x_pred[i], cmap="gray")
        ax.set_title(f"Recon {i}")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


In [3]:
X1 = load('/data/yinghuazhang/MolF-DAEs/dataset/MACCSFP_molecule3.data2')
idx = np.random.choice(X1.shape[0], 80000, replace=False)
X1 = X1[idx]

In [4]:
X2 = X1[:10]
fig = plt.figure(figsize = (20,8))
for i in range(10):
    ax = plt.subplot(2,5,i+1)
    ax.imshow(X2[i])

In [5]:
X2.shape

(10, 13, 13, 1)

In [6]:
class Encoder(Model):
    def __init__(self, dim):
        super().__init__()
        self.flatten = Flatten()
        self.d1 = Dense(512, activation='relu')
        self.d2 = Dense(1024, activation='relu')
        self.d3 = Dense(512, activation='relu')
        self.d4 = Dense(256, activation='relu')
        self.d5 = Dense(64, activation='relu')
        self.d6 = Dense(32, activation='relu')
        self.d7 = Dense(dim, activation='relu')

    def call(self, x):
        x = self.flatten(x)
        x = self.d1(x)
        x = self.d2(x)
        x = self.d3(x)
        x = self.d4(x)
        x = self.d5(x)
        x = self.d6(x)
        return self.d7(x)


class Decoder(Model):
    def __init__(self):
        super().__init__()
        self.d8 = Dense(32, activation='relu')
        self.d9 = Dense(64, activation='relu')
        self.d10 = Dense(256, activation='relu')
        self.d11 = Dense(512, activation='relu')
        self.d12 = Dense(1024, activation='relu')
        self.d13 = Dense(512, activation='relu')
        self.d14 = Dense(169, activation='sigmoid')
        self.re = Reshape((13, 13))

    def call(self, x):
        x = self.d8(x)
        x = self.d9(x)
        x = self.d10(x)
        x = self.d11(x)
        x = self.d12(x)
        x = self.d13(x)
        x = self.d14(x)
        return self.re(x)


class Autoencoder(Model):
    def __init__(self, dim):
        super().__init__()
        self.encoder = Encoder(dim)
        self.decoder = Decoder()

    def call(self, x):
        z = self.encoder(x)
        x_rec = self.decoder(z)
        return x_rec


In [7]:

# dims = [2, 3, 4, 8]
# base_model_dir = "/data/yinghuazhang/MolF-DAEs/code/control-review/model"
# base_result_dir = "/data/yinghuazhang/MolF-DAEs/code/control-review/result/maccsfp“

# ensure_dir(base_model_dir)
# ensure_dir(base_result_dir)

# # 如果 X1 不是 float32，建议先转，省显存
# X1 = X1.astype("float32")

# for dim in dims:
#     print(f"\n========== Training latent dim = {dim} ==========")
#     reset_keras()

#     # 每个维度单独目录
#     save_dir = os.path.join(base_result_dir, f"test9-{dim}D")
#     fig_dir = os.path.join(save_dir, "figures")
#     ensure_dir(save_dir)
#     ensure_dir(fig_dir)

#     # 构建模型
#     model = Autoencoder(dim=dim)

#     # 第一阶段 BCE
#     model.compile(optimizer='adam', loss='binary_crossentropy')
#     history1 = model.fit(
#         X1, X1,
#         batch_size=768,
#         epochs=100,
#         verbose=1
#     )

#     save_history_plot(
#         history1.history,
#         os.path.join(fig_dir, f"dim{dim}_stage1_bce_loss.png"),
#         title=f"Latent dim={dim}, Stage 1 BCE Loss"
#     )

#     # 第二阶段 MSE
#     model.compile(optimizer='adam', loss='mse')
#     history2 = model.fit(
#         X1, X1,
#         batch_size=768,
#         epochs=100,
#         verbose=1
#     )

#     save_history_plot(
#         history2.history,
#         os.path.join(fig_dir, f"dim{dim}_stage2_mse_loss.png"),
#         title=f"Latent dim={dim}, Stage 2 MSE Loss"
#     )

#     # 重建前10个样本并保存图片
#     y_pre = model.predict(X1[:10], verbose=0)
#     save_reconstruction_plot(
#         X1[:10],
#         y_pre,
#         os.path.join(fig_dir, f"dim{dim}_reconstruction_examples.png"),
#         n_show=10
#     )

#     # 保存模型
#     model_path = os.path.join(base_model_dir, f"maccsfp_autoencoder_dim{dim}.h5")
#     model.save(model_path)
#     print(f"Saved model to: {model_path}")

#     # 导出 latent representation
#     X_latent = model.encoder(X1).numpy()
#     latent_path = os.path.join(save_dir, "latent_vectors.joblib")
#     dump(X_latent, latent_path)
#     print(f"Saved latent vectors to: {latent_path}")

#     # 可选：保存最终重建误差
#     final_bce = history1.history["loss"][-1]
#     final_mse = history2.history["loss"][-1]
#     summary_df = pd.DataFrame({
#         "dim": [dim],
#         "final_bce_loss": [final_bce],
#         "final_mse_loss": [final_mse],
#         "latent_shape_0": [X_latent.shape[0]],
#         "latent_shape_1": [X_latent.shape[1]],
#     })
#     summary_df.to_csv(os.path.join(save_dir, "summary.csv"), index=False)

#     # 手动释放
#     del model, history1, history2, y_pre, X_latent
#     gc.collect()

# print("\nAll runs completed.")

In [ ]:
reset_keras()

# 建议先确保 X1 是 float32
X1 = X1.astype("float32")

base_model_dir = "/data/yinghuazhang/MolF-DAEs/code/control-review/model"
base_result_dir = "/data/yinghuazhang/MolF-DAEs/result/maccsfp"

ensure_dir(base_model_dir)
ensure_dir(base_result_dir)

save_dir = os.path.join(base_result_dir, f"test9-{dim}D")
fig_dir = os.path.join(save_dir, "figures")
ensure_dir(save_dir)
ensure_dir(fig_dir)

print(f"Start training dim={dim}")

model = Autoencoder(dim=dim)

# ===== Stage 1: BCE =====
model.compile(optimizer='adam', loss='binary_crossentropy')
history1 = model.fit(
    X1, X1,
    batch_size=768,
    epochs=100,
    verbose=1
)

save_history_plot(
    history1.history,
    os.path.join(fig_dir, f"dim{dim}_stage1_bce_loss.png"),
    title=f"Latent dim={dim}, Stage 1 BCE Loss"
)

# ===== Stage 2: MSE =====
model.compile(optimizer='adam', loss='mse')
history2 = model.fit(
    X1, X1,
    batch_size=768,
    epochs=100,
    verbose=1
)

save_history_plot(
    history2.history,
    os.path.join(fig_dir, f"dim{dim}_stage2_mse_loss.png"),
    title=f"Latent dim={dim}, Stage 2 MSE Loss"
)

# ===== Reconstruction examples =====
y_pre = model.predict(X1[:10], verbose=0)

save_reconstruction_plot(
    X1[:10],
    y_pre,
    os.path.join(fig_dir, f"dim{dim}_reconstruction_examples.png"),
    n_show=10
)


2026-03-06 17:57:01.737648: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 17:57:05.354066: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 19458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:21:00.0, compute capability: 8.9
2026-03-06 17:57:05.429520: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22321 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:a1:00.0, compute capability: 8.9
2026-03-06 17:57:05.430486: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created dev

5
Start training dim=3


2026-03-06 17:57:06.341020: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 19458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:21:00.0, compute capability: 8.9
2026-03-06 17:57:06.444163: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 19458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:21:00.0, compute capability: 8.9
2026-03-06 17:57:06.444736: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22321 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:a1:00.0, compute capability: 8.9
2026-03-06 17:57:06.445255: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 10098 MB memory:  -> device: 2, name: NVIDIA GeForce RT

Epoch 1/100


2026-03-06 17:57:20.036442: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


105/105 [==============================] - 16s 29ms/step - loss: 0.4196
Epoch 2/100
105/105 [==============================] - 2s 15ms/step - loss: 0.3500
Epoch 3/100
105/105 [==============================] - 1s 12ms/step - loss: 0.3053
Epoch 4/100
105/105 [==============================] - 3s 30ms/step - loss: 0.2935
Epoch 5/100
105/105 [==============================] - 4s 41ms/step - loss: 0.2876
Epoch 6/100
105/105 [==============================] - 4s 42ms/step - loss: 0.2840
Epoch 7/100
105/105 [==============================] - 6s 53ms/step - loss: 0.2818
Epoch 8/100
105/105 [==============================] - 4s 36ms/step - loss: 0.2805
Epoch 9/100
105/105 [==============================] - 2s 15ms/step - loss: 0.2788
Epoch 10/100
105/105 [==============================] - 6s 54ms/step - loss: 0.2780
Epoch 11/100
105/105 [==============================] - 6s 54ms/step - loss: 0.2756
Epoch 12/100
105/105 [==============================] - 6s 53ms/step - loss: 0.2774
Epoch 13/100

NotImplementedError: Saving the model to HDF5 format requires the model to be a Functional model or a Sequential model. It does not work for subclassed models, because such models are defined via the body of a Python method, which isn't safely serializable. Consider saving to the Tensorflow SavedModel format (by setting save_format="tf") or using `save_weights`.

In [ ]:
model_path = os.path.join(base_model_dir, f"maccsfp_autoencoder_dim{dim}")
model.save(model_path, save_format="tf")
print(f"Saved model to: {model_path}")

# ===== Save latent =====
X_latent = model.encoder(X1).numpy()
latent_path = os.path.join(save_dir, "latent_vectors.joblib")
dump(X_latent, latent_path)
print(f"Saved latent vectors to: {latent_path}")

# ===== Save history csv =====
pd.DataFrame(history1.history).to_csv(
    os.path.join(save_dir, "stage1_bce_history.csv"), index=False
)
pd.DataFrame(history2.history).to_csv(
    os.path.join(save_dir, "stage2_mse_history.csv"), index=False
)

# ===== Save summary =====
summary_df = pd.DataFrame({
    "dim": [dim],
    "final_bce_loss": [history1.history["loss"][-1]],
    "final_mse_loss": [history2.history["loss"][-1]],
    "n_samples": [X1.shape[0]]
})
summary_df.to_csv(os.path.join(save_dir, "summary.csv"), index=False)

print(f"Finished dim={dim}")

NameError: name 'os' is not defined